In [16]:
import os
from pathlib import Path

# Path lokal (macOS) atau Kaggle — otomatis pilih yang tersedia
_LOCAL = Path("/Users/jae/Developer/FINDIT/data/kaggle/search-and-rescue")
_KAGGLE = Path("/kaggle/input/datasets/nikolasgegenava/sard-search-and-rescue/search-and-rescue")
dataset_root = _KAGGLE if _KAGGLE.exists() else _LOCAL

print(f"📂 dataset_root: {dataset_root}")
print(f"   Exists: {dataset_root.exists()}")

# =============================================
# CELL 1: Eksplorasi Struktur Dataset
# =============================================

print("\n" + "=" * 50)
print("📁 STRUKTUR DATASET")
print("=" * 50)

total_images = 0
total_labels = 0
split_summary = {}

for split in ["train", "valid", "test"]:
    img_dir = dataset_root / split / "images"
    lbl_dir = dataset_root / split / "labels"

    imgs = list(img_dir.glob("*.*")) if img_dir.exists() else []
    lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

    # Cek gambar tanpa label dan sebaliknya
    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}
    missing_labels = img_stems - lbl_stems
    missing_images = lbl_stems - img_stems

    split_summary[split] = {
        "images": len(imgs),
        "labels": len(lbls),
        "missing_labels": len(missing_labels),
        "missing_images": len(missing_images)
    }

    total_images += len(imgs)
    total_labels += len(lbls)

    print(f"\n  📂 {split.upper()}")
    print(f"     Images : {len(imgs)}")
    print(f"     Labels : {len(lbls)}")
    if missing_labels:
        print(f"     ⚠️  Gambar tanpa label : {len(missing_labels)}")
    if missing_images:
        print(f"     ⚠️  Label tanpa gambar : {len(missing_images)}")

print(f"\n{'=' * 50}")
print(f"  TOTAL Images : {total_images}")
print(f"  TOTAL Labels : {total_labels}")
print(f"{'=' * 50}")

# Cek format gambar
print("\n📸 FORMAT GAMBAR:")
all_exts = {}
for split in ["train", "valid", "test"]:
    img_dir = dataset_root / split / "images"
    if img_dir.exists():
        for f in img_dir.glob("*.*"):
            ext = f.suffix.lower()
            all_exts[ext] = all_exts.get(ext, 0) + 1

for ext, count in sorted(all_exts.items(), key=lambda x: -x[1]):
    print(f"  {ext}: {count} files")


📂 dataset_root: /kaggle/input/datasets/nikolasgegenava/sard-search-and-rescue/search-and-rescue
   Exists: True

📁 STRUKTUR DATASET

  📂 TRAIN
     Images : 4041
     Labels : 4041

  📂 VALID
     Images : 1144
     Labels : 1144

  📂 TEST
     Images : 570
     Labels : 570

  TOTAL Images : 5755
  TOTAL Labels : 5755

📸 FORMAT GAMBAR:
  .jpg: 5755 files


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

# =============================================
# CELL 2: Analisis Label & Distribusi Kelas
# =============================================

print("=" * 50)
print("🏷️  ANALISIS LABEL (YOLO FORMAT)")
print("=" * 50)

class_names = ["human"]
all_annotations = {}  # {split: [{class_id, cx, cy, w, h}, ...]}
objects_per_image = {}  # {split: [count, ...]}
class_counts = {}  # {split: {class_id: count}}

for split in ["train", "valid", "test"]:
    lbl_dir = dataset_root / split / "labels"
    annotations = []
    counts_per_img = []
    cls_cnt = {}

    if lbl_dir.exists():
        for lbl_file in lbl_dir.glob("*.txt"):
            with open(lbl_file) as f:
                lines = f.read().strip().splitlines()
            n = 0
            for line in lines:
                if line.strip():
                    parts = line.strip().split()
                    cls_id = int(parts[0])
                    cx, cy, w, h = map(float, parts[1:5])
                    annotations.append({"class_id": cls_id, "cx": cx, "cy": cy, "w": w, "h": h})
                    cls_cnt[cls_id] = cls_cnt.get(cls_id, 0) + 1
                    n += 1
            counts_per_img.append(n)

    all_annotations[split] = annotations
    objects_per_image[split] = counts_per_img
    class_counts[split] = cls_cnt

    total_obj = len(annotations)
    img_with_obj = sum(1 for c in counts_per_img if c > 0)
    img_empty = sum(1 for c in counts_per_img if c == 0)
    avg_obj = np.mean(counts_per_img) if counts_per_img else 0

    print(f"\n  📂 {split.upper()}")
    print(f"     Total objek      : {total_obj}")
    print(f"     Gambar berisi obj: {img_with_obj}")
    print(f"     Gambar kosong    : {img_empty}")
    print(f"     Rata-rata obj/img: {avg_obj:.2f}")
    if cls_cnt:
        for cid, cnt in sorted(cls_cnt.items()):
            cname = class_names[cid] if cid < len(class_names) else f"class_{cid}"
            print(f"     [{cid}] {cname}: {cnt} objek")


In [ ]:
# =============================================
# CELL 3: Visualisasi Distribusi Dataset (Bar Chart)
# =============================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("📊 Distribusi Dataset Search & Rescue", fontsize=14, fontweight="bold")

splits = ["train", "valid", "test"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

# Plot 1: Jumlah gambar per split
img_counts = [split_summary[s]["images"] for s in splits]
axes[0].bar(splits, img_counts, color=colors)
axes[0].set_title("Jumlah Gambar per Split")
axes[0].set_ylabel("Jumlah")
for i, v in enumerate(img_counts):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Plot 2: Jumlah objek per split
obj_counts = [len(all_annotations[s]) for s in splits]
axes[1].bar(splits, obj_counts, color=colors)
axes[1].set_title("Jumlah Objek (Human) per Split")
axes[1].set_ylabel("Jumlah")
for i, v in enumerate(obj_counts):
    axes[1].text(i, v + 2, str(v), ha="center", fontweight="bold")

# Plot 3: Distribusi jumlah objek per gambar (train)
train_counts = objects_per_image["train"]
if train_counts:
    axes[2].hist(train_counts, bins=range(0, max(train_counts) + 2), color="#4C72B0",
                 edgecolor="white", align="left")
    axes[2].set_title("Distribusi Objek/Gambar (Train)")
    axes[2].set_xlabel("Jumlah Objek")
    axes[2].set_ylabel("Frekuensi Gambar")

plt.tight_layout()
plt.show()
print("✅ Visualisasi distribusi selesai")


In [ ]:
# =============================================
# CELL 4: Analisis Ukuran Bounding Box
# =============================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("📐 Analisis Ukuran Bounding Box (Train)", fontsize=14, fontweight="bold")

train_ann = all_annotations["train"]
if train_ann:
    widths  = [a["w"] for a in train_ann]
    heights = [a["h"] for a in train_ann]
    areas   = [a["w"] * a["h"] for a in train_ann]

    # Histogram width
    axes[0].hist(widths, bins=30, color="#4C72B0", edgecolor="white")
    axes[0].set_title("Distribusi Lebar BBox (relatif)")
    axes[0].set_xlabel("Lebar (0–1)")
    axes[0].set_ylabel("Frekuensi")
    axes[0].axvline(np.mean(widths), color="red", linestyle="--", label=f"Mean: {np.mean(widths):.3f}")
    axes[0].legend()

    # Histogram height
    axes[1].hist(heights, bins=30, color="#55A868", edgecolor="white")
    axes[1].set_title("Distribusi Tinggi BBox (relatif)")
    axes[1].set_xlabel("Tinggi (0–1)")
    axes[1].axvline(np.mean(heights), color="red", linestyle="--", label=f"Mean: {np.mean(heights):.3f}")
    axes[1].legend()

    # Scatter width vs height
    axes[2].scatter(widths, heights, alpha=0.3, s=10, color="#C44E52")
    axes[2].set_title("Lebar vs Tinggi BBox")
    axes[2].set_xlabel("Lebar (relatif)")
    axes[2].set_ylabel("Tinggi (relatif)")

    plt.tight_layout()
    plt.show()

    print(f"\n📊 Statistik BBox (Train):")
    print(f"  Lebar   — min: {min(widths):.4f}, max: {max(widths):.4f}, mean: {np.mean(widths):.4f}, median: {np.median(widths):.4f}")
    print(f"  Tinggi  — min: {min(heights):.4f}, max: {max(heights):.4f}, mean: {np.mean(heights):.4f}, median: {np.median(heights):.4f}")
    print(f"  Area    — min: {min(areas):.6f}, max: {max(areas):.6f}, mean: {np.mean(areas):.6f}")


In [ ]:
# =============================================
# CELL 5: Heatmap Posisi Center BBox
# =============================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("🗺️  Heatmap Posisi Center Bounding Box", fontsize=14, fontweight="bold")

for idx, split in enumerate(["train", "valid", "test"]):
    ann = all_annotations[split]
    if ann:
        cx_vals = [a["cx"] for a in ann]
        cy_vals = [a["cy"] for a in ann]
        h, xedges, yedges = np.histogram2d(cx_vals, cy_vals, bins=20, range=[[0,1],[0,1]])
        im = axes[idx].imshow(h.T, origin="lower", extent=[0,1,0,1],
                              cmap="hot", aspect="auto")
        axes[idx].set_title(f"{split.upper()} (n={len(ann)})")
        axes[idx].set_xlabel("Center X")
        axes[idx].set_ylabel("Center Y")
        plt.colorbar(im, ax=axes[idx])
    else:
        axes[idx].text(0.5, 0.5, "Tidak ada data", ha="center", va="center")
        axes[idx].set_title(split.upper())

plt.tight_layout()
plt.show()
print("✅ Heatmap posisi bbox selesai")


In [ ]:
# =============================================
# CELL 6: Analisis Ukuran Gambar
# =============================================

print("=" * 50)
print("🖼️  ANALISIS UKURAN GAMBAR")
print("=" * 50)

img_sizes = {}
for split in ["train", "valid", "test"]:
    img_dir = dataset_root / split / "images"
    sizes = []
    if img_dir.exists():
        img_files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        sample = random.sample(img_files, min(50, len(img_files)))
        for f in sample:
            try:
                with Image.open(f) as img:
                    sizes.append(img.size)  # (W, H)
            except Exception:
                pass
    img_sizes[split] = sizes

for split in ["train", "valid", "test"]:
    sizes = img_sizes[split]
    if sizes:
        widths  = [s[0] for s in sizes]
        heights = [s[1] for s in sizes]
        unique  = set(sizes)
        print(f"\n  📂 {split.upper()} (sample {len(sizes)} gambar)")
        print(f"     Ukuran unik : {len(unique)}")
        print(f"     Lebar  : {min(widths)}–{max(widths)} px (mean {np.mean(widths):.0f})")
        print(f"     Tinggi : {min(heights)}–{max(heights)} px (mean {np.mean(heights):.0f})")
        if len(unique) <= 3:
            print(f"     Ukuran : {unique}")


In [ ]:
# =============================================
# CELL 7: Visualisasi Sample Gambar + Bounding Box
# =============================================

def draw_bbox_on_image(img_path, lbl_path, ax, class_names=["human"]):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(img_path.name[:30], fontsize=7)

    colors_map = ["red", "blue", "green", "orange", "purple"]
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])
                x1 = (cx - w / 2) * W
                y1 = (cy - h / 2) * H
                bw = w * W
                bh = h * H
                color = colors_map[cls_id % len(colors_map)]
                rect = patches.Rectangle((x1, y1), bw, bh,
                                          linewidth=2, edgecolor=color, facecolor="none")
                ax.add_patch(rect)
                label = class_names[cls_id] if cls_id < len(class_names) else f"cls{cls_id}"
                ax.text(x1, y1 - 3, label, color=color, fontsize=8, fontweight="bold")

# Tampilkan 12 sample dari train
train_img_dir = dataset_root / "train" / "images"
train_lbl_dir = dataset_root / "train" / "labels"
img_files = list(train_img_dir.glob("*.jpg"))

# Prioritaskan gambar yang punya bounding box
labeled = [f for f in img_files if (train_lbl_dir / (f.stem + ".txt")).exists()
           and (train_lbl_dir / (f.stem + ".txt")).stat().st_size > 0]
sample_files = random.sample(labeled, min(12, len(labeled)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("🔍 Sample Gambar Train dengan Bounding Box", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, img_path in enumerate(sample_files):
    lbl_path = train_lbl_dir / (img_path.stem + ".txt")
    draw_bbox_on_image(img_path, lbl_path, axes[i])

for j in range(len(sample_files), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()
print("✅ Sample visualisasi bounding box selesai")


In [ ]:
# =============================================
# CELL 8: Ringkasan EDA
# =============================================

print("=" * 55)
print("📋  RINGKASAN EDA — SEARCH & RESCUE DATASET")
print("=" * 55)

total_obj_all = sum(len(all_annotations[s]) for s in ["train", "valid", "test"])
all_widths  = [a["w"] for s in ["train","valid","test"] for a in all_annotations[s]]
all_heights = [a["h"] for s in ["train","valid","test"] for a in all_annotations[s]]

print(f"\n  Dataset   : Search and Rescue (Roboflow)")
print(f"  Kelas     : {', '.join(class_names)} (nc=1)")
print(f"\n  ── Jumlah Data ──")
for s in ["train", "valid", "test"]:
    print(f"  {s.upper():6s} : {split_summary[s]['images']:5d} gambar | {len(all_annotations[s]):5d} objek")
print(f"  {'TOTAL':6s} : {total_images:5d} gambar | {total_obj_all:5d} objek")

print(f"\n  ── Statistik Bounding Box (semua split) ──")
if all_widths:
    print(f"  Lebar  bbox : mean={np.mean(all_widths):.4f}, std={np.std(all_widths):.4f}")
    print(f"  Tinggi bbox : mean={np.mean(all_heights):.4f}, std={np.std(all_heights):.4f}")
    small  = sum(1 for w,h in zip(all_widths,all_heights) if w*h < 0.01)
    medium = sum(1 for w,h in zip(all_widths,all_heights) if 0.01 <= w*h < 0.1)
    large  = sum(1 for w,h in zip(all_widths,all_heights) if w*h >= 0.1)
    total_boxes = len(all_widths)
    print(f"\n  ── Ukuran Objek ──")
    print(f"  Kecil  (area < 1%)   : {small:5d} ({100*small/total_boxes:.1f}%)")
    print(f"  Sedang (1%–10% area) : {medium:5d} ({100*medium/total_boxes:.1f}%)")
    print(f"  Besar  (area > 10%)  : {large:5d} ({100*large/total_boxes:.1f}%)")

print("\n" + "=" * 55)
print("✅  EDA selesai!")
print("=" * 55)
